In [29]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import joblib

# 허깅페이스에서 데이터셋 다운로드 및 로드
dataset = load_dataset("c01dsnap/CIC-IDS2017")

# 분석 및 전처리를 위해 데이터프레임으로 변환
df = dataset['train'].to_pandas()

# 데이터 확인
print(df.shape)  # (2830743, 79)
print(df[' Label'].value_counts())  # 라벨별 데이터 개수 확인

(2830743, 79)
 Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [30]:
# 공백 제거
df.columns = df.columns.str.strip()

In [31]:
df.isnull().sum()

Destination Port               0
Flow Duration                  0
Total Fwd Packets              0
Total Backward Packets         0
Total Length of Fwd Packets    0
                              ..
Idle Mean                      0
Idle Std                       0
Idle Max                       0
Idle Min                       0
Label                          0
Length: 79, dtype: int64

In [32]:
# CIC-IDS2017 데이터셋 전처리
# 공백 제거
df.columns = df.columns.str.strip()

# 무한대 값(inf, -inf)을 결측치(NaN)로 변환
df.replace([np.inf,-np.inf],np.nan,inplace=True)

# 결측치가 있는 행 제거
df.dropna( inplace=True)

# 'BENIGN'이면 0, 그 외 모든 공격은 1로 매핑
# 정상(0) : 2,271,320개
# 악성(1) : 556556개
df['Binary_Label'] = np.where(df['Label'] == 'BENIGN', 0, 1)

# 데이터 프레임에 완전 중복 열 [Fwd Header Length.1] 제거
df.drop(columns=["Fwd Header Length.1"], inplace=True)

# 제거할 열 
# 데이터의 값이 전부 0이거나 학습에 큰 영향을 미치지 않는 열
drop_cols = [
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Bulk Rate",
    "Bwd Avg Packets/Bulk",
    "Bwd PSH Flags",
    "Bwd URG Flags"
]

df.drop(columns=drop_cols, inplace=True)

In [33]:
# 2827876개 72열
print(f"최종 데이터 크기 : {df.shape}")

최종 데이터 크기 : (2827876, 71)


In [11]:
# Flow.CSV를 실시간 예측에 사용하기 위한 전처리

# Flow.CSV 파일 불러오기
flow_df = pd.read_csv("sample_120s.csv")


In [ ]:
# COLUMN_MAP을 활용해 열 이름을 CIC-IDS2017에 맞게 변경
# 현재는 사용 X
# 공백 제거
flow_df.columns = flow_df.columns.str.strip()
flow_df = flow_df.rename(columns=COLUMN_MAP)

# 학습 당시 저장한 Features 목록 불러오기
# 학습에 사용한 X와 동일한 Features
feature_columns = joblib.load("feature_columns.pkl")

# 혹시 모를 공백 제거
flow_df.columns = flow_df.columns.str.strip()

# 필요한 Feature 누락 여부 검사
missing_columns = [
    column
    for column in feature_columns
    if column not in flow_df.columns
]

print(missing_columns)
# 누락시 누락된 열 출력
if missing_columns:
    raise ValueError(
        f"Flow.csv에 학습 Feature가 누락되었습니다: {missing_columns}"
    )


# 학습에 사용한 Feature만 같은 순서로 선택
X_realtime = flow_df[feature_columns].copy()

# 숫자가 아닌 값을 NaN으로 변환
X_realtime = X_realtime.apply(
    pd.to_numeric,
    errors="coerce"
)

# 무한대 값을 NaN으로 변환
X_realtime.replace([np.inf, -np.inf], np.nan,inplace=True)

# 정상 데이터와 전처리 불가능한 데이터 구분
valid_mask = X_realtime.notna().all(axis=1)

X_realtime_valid = X_realtime.loc[valid_mask].astype(np.float32)
invalid_flows = flow_df.loc[~valid_mask].copy()

print("전체 Flow 개수:", len(flow_df))
print("예측 가능한 Flow 개수:", len(X_realtime_valid))
print("전처리 제외 Flow 개수:", len(invalid_flows))
print("모델 입력 크기:", X_realtime_valid.shape)

['Fwd Header Length.1']


ValueError: Flow.csv에 학습 Feature가 누락되었습니다: ['Fwd Header Length.1']

In [ ]:
# Feature 데이터 분리
X = df.drop(columns=["Label", "Binary_Label"])

# Label 데이터 분리
y = df["Binary_Label"]

#데이터 저장
np.save("X.npy", X.to_numpy())
np.save("y.npy", y.to_numpy())

In [35]:
# 경량화를 위해 32bit로
# Feature 데이터 분리
X = df.drop(columns=["Label", "Binary_Label"])

# Label 데이터 분리
y = df["Binary_Label"]

X = X.astype(np.float32)
np.save("X32.npy", X)
y = y.astype(np.int32)
np.save("y32.npy", y)

In [36]:
joblib.dump(X.columns.tolist(), "feature_columns.pkl")

['feature_columns.pkl']

In [34]:
# 정상과 공격 비율 최종 확인
# 0 : 정상, 1 : 공격
print(df['Binary_Label'].value_counts())

Binary_Label
0    2271320
1     556556
Name: count, dtype: int64


In [15]:
df.info()

<class 'pandas.DataFrame'>
Index: 2827876 entries, 0 to 2830742
Data columns (total 71 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  Flow IAT Min    

In [ ]:
# 중복열 개수 확인
df.duplicated().sum()

np.int64(307078)

In [ ]:
# 중복률
# 10.85% 이지만 제거하기에는 애매한 데이터
duplicate_rate = df.duplicated().mean() * 100
print(duplicate_rate)

10.858962698505875


In [ ]:
duplicates = df[df.duplicated(keep=False)]

print(duplicates.head(20))

     Destination Port  Flow Duration  Total Fwd Packets  \
76                 21             50                  1   
109                22            187                  1   
111                22            111                  1   
384               443            161                  2   
386               465             49                  2   
387                80          50486                  1   
390                80          50641                  1   
405               137             22                 13   
422                53            259                  2   
423                53            199                  2   
426                53            198                  2   
430                53            262                  2   
434                53            271                  2   
441                53            301                  2   
443               443              4                  2   
454               443            161                  2 

In [ ]:
# 중복 데이터 개수 출력
print(df.duplicated().value_counts())

False    2522362
True      308381
Name: count, dtype: int64


In [26]:
# 완전히 중복된 행 표시
duplicate_mask = df.duplicated()

# 중복 행만 추출
duplicate_rows = df.loc[duplicate_mask]

# 원본 Label별 중복 개수
print(duplicate_rows["Label"].value_counts())

Label
BENIGN                      176263
PortScan                     68110
DoS Hulk                     57278
SSH-Patator                   2678
FTP-Patator                   2004
DoS slowloris                  411
DoS Slowhttptest               271
Web Attack � Brute Force        37
DDoS                            11
Bot                              8
DoS GoldenEye                    7
Name: count, dtype: int64


In [ ]:
# 삭제해도 되는 열인지 확인
bulk_cols = [
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate"
]

print(df[bulk_cols].describe())

for col in bulk_cols:
    print(col, df[col].nunique())

# 결과를 보니 약280만개의 데이터가 0으로 제거해도 문제 없는 열

       Fwd Avg Bytes/Bulk  Fwd Avg Packets/Bulk  Fwd Avg Bulk Rate  \
count           2830743.0             2830743.0          2830743.0   
mean                  0.0                   0.0                0.0   
std                   0.0                   0.0                0.0   
min                   0.0                   0.0                0.0   
25%                   0.0                   0.0                0.0   
50%                   0.0                   0.0                0.0   
75%                   0.0                   0.0                0.0   
max                   0.0                   0.0                0.0   

       Bwd Avg Bytes/Bulk  Bwd Avg Packets/Bulk  Bwd Avg Bulk Rate  
count           2830743.0             2830743.0          2830743.0  
mean                  0.0                   0.0                0.0  
std                   0.0                   0.0                0.0  
min                   0.0                   0.0                0.0  
25%                   0.

In [ ]:
# 추가로 제거할 열 확인 
check_cols = [
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "CWE Flag Count",
    "ECE Flag Count"
]

print(df[check_cols].describe())

for col in check_cols:
    print(col, df[col].nunique())

       Bwd PSH Flags  Bwd URG Flags  CWE Flag Count  ECE Flag Count
count      2830743.0      2830743.0    2.830743e+06    2.830743e+06
mean             0.0            0.0    1.112782e-04    2.433990e-04
std              0.0            0.0    1.054826e-02    1.559935e-02
min              0.0            0.0    0.000000e+00    0.000000e+00
25%              0.0            0.0    0.000000e+00    0.000000e+00
50%              0.0            0.0    0.000000e+00    0.000000e+00
75%              0.0            0.0    0.000000e+00    0.000000e+00
max              0.0            0.0    1.000000e+00    1.000000e+00
Bwd PSH Flags 1
Bwd URG Flags 1
CWE Flag Count 2
ECE Flag Count 2


In [ ]:
# 완전 중복열 제거
df.drop(columns=["Fwd Header Length.1"], inplace=True)

In [27]:
print(df[df['CWE Flag Count'] == 1])

         Destination Port  Flow Duration  Total Fwd Packets  \
1313924               135              3                  2   
1313925               135              3                  2   
1313926             34223              3                  2   
1313927             34223              3                  2   
1313929             34223              3                  2   
...                   ...            ...                ...   
1452128             33010              3                  2   
1452129               135              3                  2   
1452130             33010             49                  2   
1458012               135             40                  1   
1458022                 1              4                  2   

         Total Backward Packets  Total Length of Fwd Packets  \
1313924                       0                            0   
1313925                       0                            0   
1313926                       0                    

In [12]:
CICIDS2017_COLUMNS = [
    "Destination Port", "Flow Duration", "Total Fwd Packets", "Total Backward Packets",
    "Total Length of Fwd Packets", "Total Length of Bwd Packets",
    "Fwd Packet Length Max", "Fwd Packet Length Min", "Fwd Packet Length Mean", "Fwd Packet Length Std",
    "Bwd Packet Length Max", "Bwd Packet Length Min", "Bwd Packet Length Mean", "Bwd Packet Length Std",
    "Flow Bytes/s", "Flow Packets/s", "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
    "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min",
    "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min",
    "Fwd PSH Flags", "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags",
    "Fwd Header Length", "Bwd Header Length", "Fwd Packets/s", "Bwd Packets/s",
    "Min Packet Length", "Max Packet Length", "Packet Length Mean", "Packet Length Std", "Packet Length Variance",
    "FIN Flag Count", "SYN Flag Count", "RST Flag Count", "PSH Flag Count", "ACK Flag Count", "URG Flag Count",
    "CWE Flag Count", "ECE Flag Count", "Down/Up Ratio", "Average Packet Size",
    "Avg Fwd Segment Size", "Avg Bwd Segment Size", "Fwd Header Length.1",
    "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",
    "Subflow Fwd Packets", "Subflow Fwd Bytes", "Subflow Bwd Packets", "Subflow Bwd Bytes",
    "Init_Win_bytes_forward", "Init_Win_bytes_backward", "act_data_pkt_fwd", "min_seg_size_forward",
    "Active Mean", "Active Std", "Active Max", "Active Min",
    "Idle Mean", "Idle Std", "Idle Max", "Idle Min",
]

COLUMN_MAP = {
    "dst_port": "Destination Port",
    "flow_duration": "Flow Duration",
    "tot_fwd_pkts": "Total Fwd Packets",
    "tot_bwd_pkts": "Total Backward Packets",
    "totlen_fwd_pkts": "Total Length of Fwd Packets",
    "totlen_bwd_pkts": "Total Length of Bwd Packets",
    "fwd_pkt_len_max": "Fwd Packet Length Max",
    "fwd_pkt_len_min": "Fwd Packet Length Min",
    "fwd_pkt_len_mean": "Fwd Packet Length Mean",
    "fwd_pkt_len_std": "Fwd Packet Length Std",
    "bwd_pkt_len_max": "Bwd Packet Length Max",
    "bwd_pkt_len_min": "Bwd Packet Length Min",
    "bwd_pkt_len_mean": "Bwd Packet Length Mean",
    "bwd_pkt_len_std": "Bwd Packet Length Std",
    "flow_byts_s": "Flow Bytes/s",
    "flow_pkts_s": "Flow Packets/s",
    "flow_iat_mean": "Flow IAT Mean",
    "flow_iat_std": "Flow IAT Std",
    "flow_iat_max": "Flow IAT Max",
    "flow_iat_min": "Flow IAT Min",
    "fwd_iat_tot": "Fwd IAT Total",
    "fwd_iat_mean": "Fwd IAT Mean",
    "fwd_iat_std": "Fwd IAT Std",
    "fwd_iat_max": "Fwd IAT Max",
    "fwd_iat_min": "Fwd IAT Min",
    "bwd_iat_tot": "Bwd IAT Total",
    "bwd_iat_mean": "Bwd IAT Mean",
    "bwd_iat_std": "Bwd IAT Std",
    "bwd_iat_max": "Bwd IAT Max",
    "bwd_iat_min": "Bwd IAT Min",
    "fwd_psh_flags": "Fwd PSH Flags",
    "bwd_psh_flags": "Bwd PSH Flags",
    "fwd_urg_flags": "Fwd URG Flags",
    "bwd_urg_flags": "Bwd URG Flags",
    "fwd_header_len": "Fwd Header Length",
    "bwd_header_len": "Bwd Header Length",
    "fwd_pkts_s": "Fwd Packets/s",
    "bwd_pkts_s": "Bwd Packets/s",
    "pkt_len_min": "Min Packet Length",
    "pkt_len_max": "Max Packet Length",
    "pkt_len_mean": "Packet Length Mean",
    "pkt_len_std": "Packet Length Std",
    "pkt_len_var": "Packet Length Variance",
    "fin_flag_cnt": "FIN Flag Count",
    "syn_flag_cnt": "SYN Flag Count",
    "rst_flag_cnt": "RST Flag Count",
    "psh_flag_cnt": "PSH Flag Count",
    "ack_flag_cnt": "ACK Flag Count",
    "urg_flag_cnt": "URG Flag Count",
    "cwr_flag_count": "CWE Flag Count",
    "ece_flag_cnt": "ECE Flag Count",
    "down_up_ratio": "Down/Up Ratio",
    "pkt_size_avg": "Average Packet Size",
    "fwd_seg_size_avg": "Avg Fwd Segment Size",
    "bwd_seg_size_avg": "Avg Bwd Segment Size",
    "fwd_byts_b_avg": "Fwd Avg Bytes/Bulk",
    "fwd_pkts_b_avg": "Fwd Avg Packets/Bulk",
    "fwd_blk_rate_avg": "Fwd Avg Bulk Rate",
    "bwd_byts_b_avg": "Bwd Avg Bytes/Bulk",
    "bwd_pkts_b_avg": "Bwd Avg Packets/Bulk",
    "bwd_blk_rate_avg": "Bwd Avg Bulk Rate",
    "subflow_fwd_pkts": "Subflow Fwd Packets",
    "subflow_fwd_byts": "Subflow Fwd Bytes",
    "subflow_bwd_pkts": "Subflow Bwd Packets",
    "subflow_bwd_byts": "Subflow Bwd Bytes",
    "init_fwd_win_byts": "Init_Win_bytes_forward",
    "init_bwd_win_byts": "Init_Win_bytes_backward",
    "fwd_act_data_pkts": "act_data_pkt_fwd",
    "fwd_seg_size_min": "min_seg_size_forward",
    "active_mean": "Active Mean",
    "active_std": "Active Std",
    "active_max": "Active Max",
    "active_min": "Active Min",
    "idle_mean": "Idle Mean",
    "idle_std": "Idle Std",
    "idle_max": "Idle Max",
    "idle_min": "Idle Min",
}